# 🏭 Visualisation Process Industriel — sans installation système

> **Aucun droits admin requis.** Le rendu graphviz s'effectue via `d3-graphviz` chargé depuis CDN (internet requis).  
> Les labels HTML, clusters par site, barres OEE/stock sont identiques à la version système.

| Fichier | Contenu |
|---|---|
| `sites.json` | Sites industriels |
| `matieres_premieres.json` | MP, stocks, seuils |
| `unites_production.json` | UP, OEE, CMJ, flux |

## 1. Imports (pip uniquement — pas de binaire système)

In [1]:
# Seul prérequis pip (pas de droits admin)
# !pip install graphviz pandas

import json, uuid
import pandas as pd
from pathlib import Path
from graphviz import Digraph          # pour GÉNÉRER le DOT source uniquement
from IPython.display import display, HTML

print('✅ Imports OK — rendu via d3-graphviz CDN (pas de dot.exe requis)')

✅ Imports OK — rendu via d3-graphviz CDN (pas de dot.exe requis)


## 2. Renderer HTML — d3-graphviz (cœur du système)

In [2]:
def show_dot(dot_obj, height=720, zoom=True):
    """
    Affiche un objet graphviz.Digraph dans Jupyter
    via d3-graphviz (CDN) — aucun binaire système requis.
    Le graphe est zoomable et déplaçable à la souris.
    """
    div_id = 'gv_' + uuid.uuid4().hex[:8]
    # Échapper les backticks et antislashs pour l'injection JS
    src = dot_obj.source.replace('\\', '\\\\').replace('`', '\\`')

    html = f"""
    <div id="{div_id}"
         style="width:100%; height:{height}px;
                background:#F8F9FA; border:1px solid #DEE2E6;
                border-radius:8px; overflow:hidden;"></div>

    <script>
    (function() {{
      function loadScript(src, cb) {{
        if (document.querySelector('script[src="' + src + '"]')) {{ cb(); return; }}
        var s = document.createElement('script');
        s.src = src; s.onload = cb;
        document.head.appendChild(s);
      }}

      var D3  = 'https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js';
      var WASM= 'https://cdn.jsdelivr.net/npm/@hpcc-js/wasm@2/dist/graphviz.umd.js';
      var GV  = 'https://cdn.jsdelivr.net/npm/d3-graphviz@5/build/d3-graphviz.min.js';

      loadScript(D3, function() {{
        loadScript(WASM, function() {{
          loadScript(GV, function() {{
            d3.select('#{div_id}')
              .graphviz()
              .zoom({'true' if zoom else 'false'})
              .fit(true)
              .renderDot(`{src}`);
          }});
        }});
      }});
    }})();
    </script>
    """
    display(HTML(html))

print('✅ show_dot() prêt')

✅ show_dot() prêt


## 3. Chargement des données

In [3]:
BASE = Path('.')

def jload(f):
    with open(BASE / f, encoding='utf-8') as fh:
        return json.load(fh)

sites = jload('sites.json')['sites']
mps   = jload('matieres_premieres.json')['matieres_premieres']
ups   = jload('unites_production.json')['unites_production']

site_map = {s['code']: s for s in sites}
mp_map   = {m['code']: m for m in mps}
up_map   = {u['code']: u for u in ups}

print(f'✅ {len(sites)} sites | {len(mps)} MP | {len(ups)} UP')

✅ 3 sites | 5 MP | 7 UP


## 4. Labels HTML — ce qui s'affiche dans chaque nœud

In [4]:
PALETTE = {
    'matiere_premiere' : ('#922B21', '#FADBD8'),
    'additif'          : ('#922B21', '#FADBD8'),
    'consommable'      : ('#922B21', '#FADBD8'),
    'emballage'        : ('#6C3483', '#E8DAEF'),
    'produit_intermediaire': ('#A04000', '#FDEBD0'),
    'produit_fini'     : ('#1A5276', '#D6EAF8'),
}

def oee_color(v):
    return '#1E8449' if v >= 85 else '#D35400' if v >= 75 else '#C0392B'

def pct(mp):
    try:    return round(mp['volume_stock'] / mp['stock_max'] * 100)
    except: return 0

def sc(mp):
    return '#C0392B' if mp['volume_stock'] <= mp['stock_min'] else \
           '#D35400' if pct(mp) < 30 else '#1E8449'

def bar(val, maxw=80, color='#2980B9'):
    """Mini barre de progression en HTML graphviz."""
    f = max(1, int(val / 100 * maxw))
    return (
        '<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">'
        f'<TR><TD BGCOLOR="{color}" WIDTH="{f}" HEIGHT="6"></TD>'
        f'<TD BGCOLOR="#D5D8DC" WIDTH="{maxw-f}" HEIGHT="6"></TD></TR>'
        '</TABLE>'
    )

# ── Label Matière première ────────────────────────────────────────────────────
def label_mp(mp):
    hbg, cbg = PALETTE.get(mp['type'], ('#922B21', '#FADBD8'))
    p  = pct(mp);  color = sc(mp)
    return (
        f'<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0" CELLPADDING="5" BGCOLOR="{cbg}">'
        f'<TR><TD COLSPAN="2" BGCOLOR="{hbg}">'
        f'<FONT COLOR="white" POINT-SIZE="10"><B>{mp["nom"]}</B></FONT></TD></TR>'
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Code</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{mp["code"]}</FONT></TD></TR>'
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Type</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{mp["type"]}</FONT></TD></TR>'
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Stock</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8" COLOR="{color}">'
        f'<B>{mp["volume_stock"]:,} {mp["unite_volume"]}</B></FONT></TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">{bar(p, color=color)}</TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">'
        f'<FONT POINT-SIZE="7" COLOR="{color}">{p}% du stock max — '
        f'min {mp["stock_min"]:,}</FONT></TD></TR>'
        f'</TABLE>>'
    )

# ── Label Unité de production ─────────────────────────────────────────────────
def label_up(up):
    hbg, cbg = PALETTE.get(up['type_produit'], ('#1A5276', '#D6EAF8'))
    oc   = oee_color(up['oee'])
    site = site_map.get(up['site_code'], {}).get('nom', up['site_code'])
    taux = round(up['cmj'] / up['capacite_max_j'] * 100)
    brd  = '3' if up['type_ligne'] == 'principale' else '1'
    return (
        f'<<TABLE BORDER="{brd}" CELLBORDER="1" CELLSPACING="0" CELLPADDING="5" BGCOLOR="{cbg}">'
        # Nom
        f'<TR><TD COLSPAN="2" BGCOLOR="{hbg}">'
        f'<FONT COLOR="white" POINT-SIZE="11"><B>{up["nom"]}</B></FONT></TD></TR>'
        # Produit
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">'
        f'<FONT POINT-SIZE="9">&#128230; {up["produit"]}</FONT></TD></TR>'
        # Site / code
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Site</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8"><B>{site}</B></FONT></TD></TR>'
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Code</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{up["code"]} '
        f'| {up["type_ligne"]}</FONT></TD></TR>'
        # Séparateur
        f'<TR><TD COLSPAN="2" BGCOLOR="{hbg}" HEIGHT="2"></TD></TR>'
        # OEE
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="9"><B>OEE</B></FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="9" COLOR="{oc}">'
        f'<B>{up["oee"]}%</B></FONT></TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">{bar(up["oee"], color=oc)}</TD></TR>'
        # CMJ
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">CMJ</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{up["cmj"]:,} '
        f'{up["unite_capacite"]}</FONT></TD></TR>'
        # Cap max
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Cap. max/j</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8">{up["capacite_max_j"]:,}</FONT></TD></TR>'
        # Taux charge
        f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8">Taux charge</FONT></TD>'
        f'<TD ALIGN="RIGHT"><FONT POINT-SIZE="8"><B>{taux}%</B></FONT></TD></TR>'
        f'<TR><TD COLSPAN="2" ALIGN="CENTER">{bar(taux)}</TD></TR>'
        f'</TABLE>>'
    )

print('✅ Labels OK')

✅ Labels OK


## 5. Construction du graphe DOT

In [5]:
def build_graph(
    show_mp      = True,
    site_filter  = None,
    rankdir      = 'LR',
    clusters     = True,
):
    dot = Digraph(
        name='process',
        graph_attr=dict(
            rankdir  = rankdir,
            splines  = 'ortho',
            nodesep  = '0.7',
            ranksep  = '1.2',
            bgcolor  = '#F8F9FA',
            fontname = 'Helvetica',
            label    = 'PROCESS INDUSTRIEL MULTI-SITES',
            labelloc = 't',
            fontsize = '18',
            fontcolor= '#2C3E50',
        ),
        node_attr=dict(shape='plaintext', fontname='Helvetica'),
        edge_attr=dict(fontname='Helvetica', fontsize='8'),
    )

    ups_f = [u for u in ups
             if site_filter is None or u['site_code'] == site_filter]
    up_codes = {u['code'] for u in ups_f}

    # Nœuds UP — par cluster site
    if clusters:
        for s in sites:
            site_ups = [u for u in ups_f if u['site_code'] == s['code']]
            if not site_ups:
                continue
            with dot.subgraph(name=f'cluster_{s["code"]}') as sg:
                sg.attr(
                    label     = f'{s["nom"]}  •  {s["localisation"]}',
                    style     = 'rounded,filled',
                    fillcolor = s.get('couleur','#4A90D9') + '18',
                    color     = s.get('couleur','#2C3E50'),
                    fontsize  = '12',
                    fontcolor = s.get('couleur','#2C3E50'),
                    penwidth  = '2',
                )
                for u in site_ups:
                    sg.node(u['code'], label=label_up(u))
    else:
        for u in ups_f:
            dot.node(u['code'], label=label_up(u))

    # Nœuds MP
    if show_mp:
        for mp in mps:
            if site_filter and mp['site_code'] != site_filter:
                continue
            dot.node(mp['code'], label=label_mp(mp))

    # Arêtes MP → UP
    if show_mp:
        for u in ups_f:
            for src in u.get('matieres_premieres_aval', []):
                if src.startswith('MP-') and src in mp_map:
                    mp = mp_map[src]
                    if site_filter and mp['site_code'] != site_filter:
                        continue
                    dot.edge(src, u['code'],
                             color='#C0392B', style='dashed',
                             penwidth='1.5', arrowsize='0.8',
                             label=mp['nom'][:14])

    # Arêtes UP → UP
    for u in ups_f:
        for nxt in u.get('produits_suivants', []):
            if nxt not in {x['code'] for x in ups}:
                continue
            if site_filter and nxt not in up_codes:
                continue
            main = u['type_ligne'] == 'principale'
            dot.edge(u['code'], nxt,
                     color    = '#2C3E50' if main else '#7F8C8D',
                     penwidth = '3' if main else '1.5',
                     style    = 'solid' if main else 'dashed',
                     arrowsize= '1.0' if main else '0.7',
                     label    = u['produit'][:20] if main else '',
                     fontcolor= '#2C3E50')
    return dot

print('✅ build_graph() prête')

✅ build_graph() prête


## 6. 🗺️ Vue complète — tous sites
*Utilisez la molette pour zoomer, clic-glisser pour déplacer*

In [6]:
g = build_graph(show_mp=True, rankdir='LR')
show_dot(g, height=750)

## 7. ⚙️ Flux principaux — sans matières premières

In [7]:
g2 = build_graph(show_mp=False, rankdir='TB')
show_dot(g2, height=650)

## 8. 🔍 Vue par site

In [8]:
for s in sites:
    print(f"  {s['code']}  →  {s['nom']} ({s['localisation']})")

# ← Modifier le site ici
g3 = build_graph(show_mp=True, site_filter='SITE-A', rankdir='TB')
show_dot(g3, height=600)

  ROC  →  Les Roches de Condrieu (Lyon)
  BGS  →  Usine de Burgos (Burgos)
  RON  →  Roussillon (Roussillon)


## 9. 💾 Export — génère le fichier .dot (ouvrable sur graphviz.org)

In [9]:
# Sauvegarde le source DOT — pas besoin de droits admin
g_export = build_graph(show_mp=True, rankdir='LR')

with open('process_industriel.dot', 'w', encoding='utf-8') as f:
    f.write(g_export.source)

print('✅ process_industriel.dot sauvegardé')
print()
print('Pour le convertir en image sans droits admin :')
print('  → Coller le contenu sur https://dreampuf.github.io/GraphvizOnline/')
print('  → Ou https://edotor.net  (export PNG/SVG/PDF directement)')

✅ process_industriel.dot sauvegardé

Pour le convertir en image sans droits admin :
  → Coller le contenu sur https://dreampuf.github.io/GraphvizOnline/
  → Ou https://edotor.net  (export PNG/SVG/PDF directement)


---
## 💡 Référence rapide

| Action | Code |
|---|---|
| Vue complète | `build_graph()` |
| Sans MP | `build_graph(show_mp=False)` |
| Haut→bas | `build_graph(rankdir='TB')` |
| Un seul site | `build_graph(site_filter='SITE-B')` |
| Sans clusters | `build_graph(clusters=False)` |
| Zoom/déplacement | Molette + clic-glisser dans la cellule |
| Ajouter un nœud | Éditer le JSON correspondant |
| Modifier un label | Fonctions `label_mp()` / `label_up()` |